# 01. Khám Phá & Đánh Giá Dữ Liệu (Exploratory Data Analysis - EDA)

- **Đồ án:** Tối ưu chi phí bảo trì dự đoán bằng Machine Learning
- **Giai đoạn:** Tuần 1 - Giai đoạn 2 (W1-G2-A)
- **Tác giả phụ trách:** Antigravity (theo phân quyền `AGENTS.md`)
- **Dữ liệu nguồn:** UCI AI4I 2020 Predictive Maintenance Dataset (DOI: `10.24432/C5HS5C`)
- **Trạng thái:** Dữ liệu raw local đã tải và xác thực hash SHA-256. Không sử dụng test set để tuning hay chọn trước mô hình (tuân thủ protocol).


In [1]:
import os
import sys
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm đường dẫn project root để import module src
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.visualization import (
    set_plot_style,
    plot_target_and_subfailures,
    plot_numerical_distributions,
    plot_boxplots_by_target,
    plot_correlation_matrix,
    plot_physical_mechanisms,
    plot_temporal_patterns,
)

set_plot_style()
print("Môi trường và các thư viện đã sẵn sàng.")


Môi trường và các thư viện đã sẵn sàng.


## 1. Xác thực nguồn gốc dữ liệu & Provenance

Kiểm tra tính toàn vẹn của tệp dữ liệu raw `data/raw/ai4i2020.csv` bằng mã băm mật mã học SHA-256 đối chiếu với thông tin tải về từ UCI ML Repository.


In [2]:
data_path = os.path.join("..", "data", "raw", "ai4i2020.csv")

if not os.path.exists(data_path):
    data_path = os.path.join("data", "raw", "ai4i2020.csv")

with open(data_path, "rb") as f:
    file_bytes = f.read()
    sha256_hash = hashlib.sha256(file_bytes).hexdigest()

file_size_bytes = len(file_bytes)

print(f"Đường dẫn file: {data_path}")
print(f"Kích thước file: {file_size_bytes:,} bytes")
print(f"SHA-256 Hash: {sha256_hash}")

expected_hash = "dc6630cd9b1f0f853922fad78a1b6436570d3f1ec863f1dd5c4340ac56bc8a8e"
assert sha256_hash.lower() == expected_hash.lower(), "Hash không khớp với bản tải gốc!"
print("=> Xác thực toàn vẹn dữ liệu: HOÀN TOÀN TRÙNG KHỚP.")


Đường dẫn file: ..\data\raw\ai4i2020.csv
Kích thước file: 522,048 bytes
SHA-256 Hash: dc6630cd9b1f0f853922fad78a1b6436570d3f1ec863f1dd5c4340ac56bc8a8e
=> Xác thực toàn vẹn dữ liệu: HOÀN TOÀN TRÙNG KHỚP.


## 2. Khảo sát Schema, Kiểu dữ liệu, Missing & Duplicates

Kiểm tra chi tiết kích thước dữ liệu, kiểu dữ liệu thực tế của từng cột, kiểm tra giá trị thiếu (missing values) và hàng trùng lặp (duplicates).


In [3]:
df = pd.read_csv(data_path)
print(f"Kích thước tập dữ liệu (Hàng, Cột): {df.shape}")
print("-" * 60)
print(df.info())
print("-" * 60)

missing_counts = df.isnull().sum()
print("Số lượng giá trị Missing/NaN trên từng cột:")
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "Không có bất kỳ giá trị NaN/Missing nào!")

# Kiểm tra duplicates
full_dups = df.duplicated().sum()
dups_no_id = df.drop(columns=["UDI", "Product ID"]).duplicated().sum()
print(f"Số dòng trùng lặp hoàn toàn (toàn bộ cột): {full_dups}")
print(f"Số dòng trùng lặp đặc trưng (bỏ qua UDI và Product ID): {dups_no_id}")


Kích thước tập dữ liệu (Hàng, Cột): (10000, 14)
------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 no

## 3. Khảo sát Nhãn mục tiêu (`Machine failure`) & 5 Chế độ Hỏng hóc Phụ

- Tỷ lệ mất cân bằng lớp (Class Imbalance): Nhãn `Machine failure` có tỷ lệ dương tính rất thấp (~3.39%).
- Năm chế độ hỏng hóc độc lập:
  1. `TWF` (Tool Wear Failure): Hỏng do mòn dụng cụ.
  2. `HDF` (Heat Dissipation Failure): Hỏng do tản nhiệt kém.
  3. `PWF` (Power Failure): Hỏng do công suất nằm ngoài dải an toàn.
  4. `OSF` (Overstrain Failure): Hỏng do ứng suất quá tải.
  5. `RNF` (Random Failure): Hỏng hóc ngẫu nhiên (xác suất 0.1%).

### Phát hiện bất thường then chốt về nhãn:
1. Có **9 quan sát** mà `Machine failure == 1` nhưng toàn bộ 5 nhãn phụ đều bằng 0 (`TWF=HDF=PWF=OSF=RNF=0`).
2. Có **18 quan sát** mà `RNF == 1` nhưng `Machine failure == 0`. Chỉ duy nhất 1 quan sát có `RNF == 1` và `Machine failure == 1` (do đồng thời xảy ra `TWF == 1`).
3. Có **24 quan sát** xảy ra từ 2 chế độ hỏng hóc trở lên đồng thời (multi-failure).


In [4]:
target_counts = df["Machine failure"].value_counts()
target_pct = df["Machine failure"].value_counts(normalize=True) * 100
summary_target = pd.DataFrame({"Số lượng": target_counts, "Tỷ lệ (%)": target_pct})
print("=== PHÂN BỐ NHÃN MỤC TIÊU (Machine failure) ===")
print(summary_target)

sub_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
print("\n=== SỐ LƯỢNG QUAN SÁT TỪNG CHẾ ĐỘ HỎNG HÓC ===")
sub_counts = df[sub_cols].sum()
print(sub_counts)

# Phân tích quan hệ logic
df_analysis = df.copy()
df_analysis["any_sub_failure"] = (df_analysis[sub_cols].sum(axis=1) > 0).astype(int)
df_analysis["sum_sub_failures"] = df_analysis[sub_cols].sum(axis=1)

print("\n=== BẢNG CHÉO: Machine failure vs Ít nhất một nhãn phụ bật ===")
print(pd.crosstab(df_analysis["Machine failure"], df_analysis["any_sub_failure"], rownames=["Machine failure"], colnames=["any_sub_failure"]))

mismatch_9 = df_analysis[(df_analysis["Machine failure"] == 1) & (df_analysis["any_sub_failure"] == 0)]
print(f"\nSố dòng Machine failure = 1 nhưng không có nhãn phụ nào: {len(mismatch_9)}")
print("Danh sách UDI của 9 dòng này:", mismatch_9["UDI"].tolist())

mismatch_18 = df_analysis[(df_analysis["Machine failure"] == 0) & (df_analysis["any_sub_failure"] == 1)]
print(f"\nSố dòng Machine failure = 0 nhưng có ít nhất một nhãn phụ bật: {len(mismatch_18)}")
print("Phân bố chế độ hỏng trong 18 dòng này:\n", mismatch_18[sub_cols].sum())


=== PHÂN BỐ NHÃN MỤC TIÊU (Machine failure) ===
                 Số lượng  Tỷ lệ (%)
Machine failure                     
0                    9661      96.61
1                     339       3.39

=== SỐ LƯỢNG QUAN SÁT TỪNG CHẾ ĐỘ HỎNG HÓC ===
TWF     46
HDF    115
PWF     95
OSF     98
RNF     19
dtype: int64

=== BẢNG CHÉO: Machine failure vs Ít nhất một nhãn phụ bật ===
any_sub_failure     0    1
Machine failure           
0                9643   18
1                   9  330

Số dòng Machine failure = 1 nhưng không có nhãn phụ nào: 9
Danh sách UDI của 9 dòng này: [1438, 2750, 4045, 4685, 5537, 5942, 6479, 8507, 9016]

Số dòng Machine failure = 0 nhưng có ít nhất một nhãn phụ bật: 18
Phân bố chế độ hỏng trong 18 dòng này:
 TWF     0
HDF     0
PWF     0
OSF     0
RNF    18
dtype: int64


In [5]:
fig_target = plot_target_and_subfailures(df)
plt.show()


C:\Users\Huỳnh Phúc\AppData\Local\Temp\ipykernel_30556\1707951880.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Phân tích Phân phối Đặc trưng Số học & Phân loại (`Type`)

- `Type`: Phân bố sản phẩm L (60%), M (30%), H (10%). Tỷ lệ hỏng giảm dần theo cấp chất lượng: $L (3.92\%) > M (2.77\%) > H (2.09\%)$.
- Các biến liên tục: Nhiệt độ môi trường, Nhiệt độ quá trình, Vận tốc quay, Mô-men xoắn, Mòn dụng cụ.


In [6]:
num_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

stats_table = df[num_cols].describe().T
stats_table["median"] = df[num_cols].median()
stats_table["skewness"] = df[num_cols].skew()
print("=== THỐNG KÊ MÔ TẢ CÁC BIẾN LIÊN TỤC ===")
print(stats_table[["mean", "std", "min", "25%", "median", "75%", "max", "skewness"]].round(3))

print("\n=== TỶ LỆ HỎNG HÓC PHÂN THEO LOẠI SẢN PHẨM (Type) ===")
type_summary = df.groupby("Type")["Machine failure"].agg(
    Tong_So_Quan_Sat="count",
    So_Ca_Hong="sum",
    Ty_Le_Hong="mean",
)
type_summary["Ty_Le_Hong (%)"] = type_summary["Ty_Le_Hong"] * 100
print(type_summary.round(3))


=== THỐNG KÊ MÔ TẢ CÁC BIẾN LIÊN TỤC ===
                             mean      std     min     25%  median     75%  \
Air temperature [K]       300.005    2.000   295.3   298.3   300.1   301.5   
Process temperature [K]   310.006    1.484   305.7   308.8   310.1   311.1   
Rotational speed [rpm]   1538.776  179.284  1168.0  1423.0  1503.0  1612.0   
Torque [Nm]                39.987    9.969     3.8    33.2    40.1    46.8   
Tool wear [min]           107.951   63.654     0.0    53.0   108.0   162.0   

                            max  skewness  
Air temperature [K]       304.5     0.114  
Process temperature [K]   313.8     0.015  
Rotational speed [rpm]   2886.0     1.993  
Torque [Nm]                76.6    -0.010  
Tool wear [min]           253.0     0.027  

=== TỶ LỆ HỎNG HÓC PHÂN THEO LOẠI SẢN PHẨM (Type) ===
      Tong_So_Quan_Sat  So_Ca_Hong  Ty_Le_Hong  Ty_Le_Hong (%)
Type                                                          
H                 1003          21       0.02

In [7]:
fig_dist = plot_numerical_distributions(df)
plt.show()


C:\Users\Huỳnh Phúc\AppData\Local\Temp\ipykernel_30556\4032338409.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Phân tích Outlier & Ý nghĩa Miền Nghiệp vụ

Đánh giá điểm ngoại lai theo phương pháp IQR ($[Q_1 - 1.5 \times IQR, Q_3 + 1.5 \times IQR]$) và quy tắc 3-sigma.

**Ý nghĩa nghiệp vụ quan trọng:**
- `Rotational speed [rpm]`: 418 điểm ngoại lai phía trên (skewness = 1.99), tỷ lệ hỏng trong nhóm này là 8.37% (cao gấp 2.5 lần trung bình toàn tập).
- `Torque [Nm]`: 64 điểm ngoại lai theo IQR. Trong 64 điểm này, có tới **58 ca hỏng hóc** (tỷ lệ hỏng = **90.625%**)!
- **Kết luận:** Điểm ngoại lai không phải là lỗi thu thập số liệu hay nhiễu kỹ thuật mà là **chỉ báo vật lý quyết định của trạng thái hư hỏng**. Tuyệt đối **không được xóa bỏ hay cắt tỉa (clipping)** tùy tiện trong pipeline tiền xử lý!


In [8]:
print("=== PHÂN TÍCH NGOẠI LAI (IQR & 3-SIGMA) ===")
for col in num_cols:
    s = df[col]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    iqr_mask = (s < lower_bound) | (s > upper_bound)
    n_iqr = iqr_mask.sum()
    pct_iqr = (n_iqr / len(df)) * 100
    
    fail_rate_in_outliers = df.loc[iqr_mask, "Machine failure"].mean() * 100 if n_iqr > 0 else 0.0
    
    mean_val = s.mean()
    std_val = s.std()
    n_3sigma = ((s < mean_val - 3 * std_val) | (s > mean_val + 3 * std_val)).sum()
    
    print(f"{col:25s} | IQR bounds: [{lower_bound:7.2f}, {upper_bound:7.2f}] | Outliers: {n_iqr:4d} ({pct_iqr:5.2f}%) | Tỷ lệ hỏng: {fail_rate_in_outliers:5.2f}% | 3-sigma: {n_3sigma:3d}")


=== PHÂN TÍCH NGOẠI LAI (IQR & 3-SIGMA) ===
Air temperature [K]       | IQR bounds: [ 293.50,  306.30] | Outliers:    0 ( 0.00%) | Tỷ lệ hỏng:  0.00% | 3-sigma:   0
Process temperature [K]   | IQR bounds: [ 305.35,  314.55] | Outliers:    0 ( 0.00%) | Tỷ lệ hỏng:  0.00% | 3-sigma:   0
Rotational speed [rpm]    | IQR bounds: [1139.50, 1895.50] | Outliers:  418 ( 4.18%) | Tỷ lệ hỏng:  8.37% | 3-sigma: 164
Torque [Nm]               | IQR bounds: [  12.80,   67.20] | Outliers:   69 ( 0.69%) | Tỷ lệ hỏng: 89.86% | 3-sigma:  25
Tool wear [min]           | IQR bounds: [-110.50,  325.50] | Outliers:    0 ( 0.00%) | Tỷ lệ hỏng:  0.00% | 3-sigma:   0


In [9]:
fig_box = plot_boxplots_by_target(df)
plt.show()


C:\Users\Huỳnh Phúc\AppData\Local\Temp\ipykernel_30556\956745901.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Phân tích Tương quan & Cơ chế Vật lý Sinh Dữ Liệu

Kiểm chứng các công thức xác định (deterministic rules) được tác giả Stephan Matzka (2020) sử dụng khi mô phỏng tập dữ liệu AI4I:
1. **HDF (Heat Dissipation Failure):** Xảy ra khi chênh lệch nhiệt độ $\Delta T = T_{process} - T_{air} < 8.6\text{ K}$ VÀ vận tốc quay $v < 1380\text{ rpm}$. (Khớp 100% với cột `HDF`).
2. **PWF (Power Failure):** Công suất cơ học $P = \tau \cdot \omega = \text{Torque} \cdot \frac{2\pi \cdot \text{Speed}}{60} < 3500\text{ W}$ HOẶC $> 9000\text{ W}$. (Khớp 100% với cột `PWF`).
3. **OSF (Overstrain Failure):** Tích mài mòn và mô-men $\text{Tool wear} \times \text{Torque} >$ ngưỡng (11,000 với L, 12,000 với M, 13,000 với H). (Khớp 100% với cột `OSF`).


In [10]:
# Kiểm chứng công thức HDF
temp_diff = df["Process temperature [K]"] - df["Air temperature [K]"]
hdf_rule = (temp_diff < 8.6) & (df["Rotational speed [rpm]"] < 1380)
print(f"Khớp tuyệt đối quy tắc vật lý HDF: {(hdf_rule == df['HDF']).all()} (Số ca: {hdf_rule.sum()})")

# Kiểm chứng công thức PWF
power_w = df["Torque [Nm]"] * (df["Rotational speed [rpm]"] * 2 * np.pi / 60)
pwf_rule = (power_w < 3500) | (power_w > 9000)
print(f"Khớp tuyệt đối quy tắc vật lý PWF: {(pwf_rule == df['PWF']).all()} (Số ca: {pwf_rule.sum()})")

# Kiểm chứng công thức OSF
strain = df["Tool wear [min]"] * df["Torque [Nm]"]
osf_thresh = df["Type"].map({"L": 11000, "M": 12000, "H": 13000})
osf_rule = strain > osf_thresh
print(f"Khớp tuyệt đối quy tắc vật lý OSF: {(osf_rule == df['OSF']).all()} (Số ca: {osf_rule.sum()})")


Khớp tuyệt đối quy tắc vật lý HDF: True (Số ca: 115)
Khớp tuyệt đối quy tắc vật lý PWF: True (Số ca: 95)
Khớp tuyệt đối quy tắc vật lý OSF: True (Số ca: 98)


In [11]:
fig_corr = plot_correlation_matrix(df)
plt.show()

fig_mech = plot_physical_mechanisms(df)
plt.show()


C:\Users\Huỳnh Phúc\AppData\Local\Temp\ipykernel_30556\435047792.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Huỳnh Phúc\AppData\Local\Temp\ipykernel_30556\435047792.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Khảo sát Thứ tự Quan sát (Sequence), Chu kỳ Nhiệt & Cụm Lỗi HDF

- Thứ tự chỉ số quan sát `UDI` (1 đến 10,000) phản ánh quá trình mô phỏng liên tục có biến thiên nhiệt độ môi trường dạng chu kỳ (thermal drift / cycle).
- Cụm nhiệt độ cao tập trung vào khoảng UDI 3000–5000:
  - 108 trong tổng số 115 ca HDF nằm gọn trong đoạn UDI 4000–4999 (và 7 ca nằm ở UDI 3000–3999).
  - Toàn bộ các đoạn khác (UDI 0–2999 và 5000–10000) có **0 ca HDF**.
- **Cảnh báo Data Leakage & Hướng dẫn Chia tập dữ liệu (Data Split):**
  - Nếu chia tập theo thứ tự thời gian tuần tự (time-based sequential split, ví dụ 60/20/20): Tập test (UDI 8000–10000) sẽ không chứa bất kỳ ca hỏng hóc HDF nào, dẫn tới đánh giá sai lệch hoàn toàn.
  - Khuyến nghị chia dữ liệu: Sử dụng **Stratified Random Split** (ví dụ 60% Train, 20% Validation, 20% Test phân tầng theo `Machine failure`), đảm bảo các chế độ hỏng hóc được phân bố đều đặn.
  - Tuyệt đối cấm sử dụng `UDI` và `Product ID` làm đặc trưng đầu vào cho mô hình ML.


In [12]:
fig_temp = plot_temporal_patterns(df)
plt.show()


C:\Users\Huỳnh Phúc\AppData\Local\Temp\ipykernel_30556\2415346782.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Kết luận Kiểm tra Dữ liệu & Đề xuất cho Data Pipeline (Codex - G3)

### 8.1. Danh mục đặc trưng giữ lại cho mô hình:
- `Type` (categorical: L, M, H -> one-hot encoding hoặc ordinal encoding $L=0, M=1, H=2$).
- `Air temperature [K]` (numerical continuous).
- `Process temperature [K]` (numerical continuous).
- `Rotational speed [rpm]` (numerical continuous, skewness cao).
- `Torque [Nm]` (numerical continuous).
- `Tool wear [min]` (numerical continuous).

### 8.2. Danh mục cột TUYỆT ĐỐI LOẠI BỎ (Chống Target Leakage & ID Artifacts):
- `UDI`: Chỉ số thứ tự hàng, gây spurious correlation.
- `Product ID`: Mã định danh sản phẩm.
- `TWF`, `HDF`, `PWF`, `OSF`, `RNF`: Đây là **chế độ hỏng hóc thứ cấp (nguyên nhân sự cố)**, chỉ xuất hiện sau khi hỏng hóc xảy ra. Đưa vào làm feature sẽ gây rò rỉ nhãn mục tiêu tuyệt đối (Data Leakage).

### 8.3. Đề xuất Feature Engineering cho Codex (khi được duyệt):
- $\Delta T = \text{Process temperature [K]} - \text{Air temperature [K]}$
- Công suất cơ học xấp xỉ: $P = \text{Torque [Nm]} \times \text{Rotational speed [rpm]} \times \frac{2\pi}{60}$
- Ứng suất mài mòn tích lũy: $\text{Overstrain} = \text{Tool wear [min]} \times \text{Torque [Nm]}$
